## DS 4420 - Proof of Concept (Model Training)

- Key Question: to what extent can a game's content, quality, and engagement metrics be used to predict its price on the Steam marketplace?" 

- File purpose: Model Training for POC

- Date: 03/13

- Authors: Gianna Saw, Jason Zheng 


In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
# pulling the df we cleaned earlier 
steam = pd.read_csv('steam_clean.csv')
steam

,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,Windows,Mac,Linux,Metacritic score,User score,Positive,Negative,Achievements,Recommendations,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,language_count
0,0.000000,0.0,5.24,4.189655,0,1,0,0,0,0,5.533389,1.386294,0,5.446737,2.197225,0.000000,2.197225,0.000000,0
1,2.197225,0.0,35.99,2.397895,1,1,0,0,0,0,4.770685,2.639057,0,4.682131,6.516193,0.000000,7.134094,0.000000,0
2,0.000000,0.0,1.49,4.262680,0,1,1,0,0,0,4.174387,1.098612,13,0.000000,0.000000,0.000000,0.000000,0.000000,0
3,0.000000,0.0,1.35,4.204693,0,1,0,0,0,0,3.555348,2.397895,5,0.000000,0.000000,0.000000,0.000000,0.000000,0
4,4.174387,0.0,2.99,4.394449,1,1,1,0,0,0,9.338558,6.723832,11,9.328212,5.837730,4.465908,5.159055,4.941642,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40672,0.693147,0.0,4.74,4.330733,1,1,1,1,0,0,5.361292,3.713572,31,5.420535,4.454347,0.000000,4.521789,0.000000,0
40673,0.000000,0.0,0.99,3.931826,0,1,0,0,0,0,5.267858,2.564949,20,5.220356,0.000000,0.000000,0.000000,0.000000,0
40674,4.382027,0.0,5.35,3.526361,0,1,1,1,0,0,8.087333,5.913503,8,8.421563,5.598422,0.000000,5.484797,0.000000,0
40675,0.000000,0.0,2.49,3.931826,0,1,1,1,0,0,5.342334,3.091042,4,5.379897,0.000000,0.000000,0.000000,0.000000,0


In [3]:
steam.shape

(40677, 19)

Splitting our train/test data + Normalising

In [4]:
# shuffling 
steam = steam.sample(frac=1, random_state=42).reset_index(drop=True)

# splitting our data (80/20)
split = int(0.8 * len(steam))
train = steam.iloc[:split]
test = steam.iloc[split:]

In [5]:
# sep features and target 
X_train = train.drop(columns=['Price']).values
y_train = train['Price'].values.reshape(-1, 1)

X_test = test.drop(columns=['Price']).values
y_test = test['Price'].values.reshape(-1, 1)

In [6]:
# min max scaling using our training data only
X_min = X_train.min(axis=0)
X_max = X_train.max(axis=0)

In [7]:
X_train_scaled = (X_train - X_min) / (X_max - X_min + 1e-8)
X_test_scaled = (X_test - X_min) / (X_max - X_min + 1e-8)

In [ ]:
# adding bias column
X_train_scaled = np.hstack([X_train_scaled, np.ones((X_train_scaled.shape[0], 1))])
X_test_scaled = np.hstack([X_test_scaled, np.ones((X_test_scaled.shape[0], 1))])

print(X_train_scaled.shape)  

(32541, 19)


MLP Implementation

In [10]:
## MLP setup

eta = 0.01
epochs = 500
p = X_train.shape[1]  # 19 (18 features + bias)
H = 4                  # hidden units

np.random.seed(1)
W1 = np.random.normal(0, 0.1, (p, H))
np.random.seed(1)
w2 = np.random.normal(0, 0.1, (H, 1))

In [11]:
# activation function
def relu(z):
    return np.maximum(0, z)


In [12]:
# gradient descent
errors = []

for epoch in range(epochs):
    grad_w2 = np.zeros_like(w2)
    grad_W1 = np.zeros_like(W1)
    loss = 0

    for i in range(X_train.shape[0]):
        x = X_train[i].reshape(-1, 1)
        yi = y_train[i]

        # forward pass
        # hidden layer
        h = relu(W1.T @ x)      
        f = w2.T @ h              

        # MSE loss
        loss += (f - yi) ** 2

        # gradients
        grad_w2 += (f - yi) * h
        indicator = (W1.T @ x > 0).astype(int)
        grad_W1 += (f - yi) * (x @ (w2 * indicator).T)

    # average gradients
    grad_w2 /= X_train.shape[0]
    grad_W1 /= X_train.shape[0]

    # update weights
    w2 -= eta * grad_w2
    W1 -= eta * grad_W1

    errors.append(loss / X_train.shape[0])

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, MSE Loss: {float(loss/X_train.shape[0]):.4f}")


/var/folders/fh/x26501n92rx17tf3wj0z4llr0000gn/T/ipykernel_32294/194331899.py:37: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print(f"Epoch {epoch}, MSE Loss: {float(loss/X_train.shape[0]):.4f}")


Epoch 0, MSE Loss: 39.9209
Epoch 50, MSE Loss: 11.7058
Epoch 100, MSE Loss: 11.3238
Epoch 150, MSE Loss: 11.0918
Epoch 200, MSE Loss: 10.9148
Epoch 250, MSE Loss: 17.2582
Epoch 300, MSE Loss: 13.4150
Epoch 350, MSE Loss: 12.4880
Epoch 400, MSE Loss: 12.0194
Epoch 450, MSE Loss: 11.6773


In [17]:
# test predicts
preds = []
for i in range(X_test.shape[0]):
    x = X_test[i].reshape(-1, 1)
    h = relu(W1.T @ x)
    f = w2.T @ h
    preds.append(float(f))

preds = np.array(preds)


/var/folders/fh/x26501n92rx17tf3wj0z4llr0000gn/T/ipykernel_32294/185759057.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  preds.append(float(f))


In [18]:
# RMSE 
rmse = np.sqrt(np.mean((preds - y_test) ** 2))
print(f"Test RMSE (log scale): {rmse:.4f}")

Test RMSE (log scale): 3.3899


Future steps: 
- Sample-by-sample loop has been quite slow on 32k rows. 
- For Phase II, we were thinking of vectorising it / looking for other ways to optimise run times and make model better because current errors are pretty high.